In [1]:
from datasets import load_dataset
ds = load_dataset("eturok-weizmann/laser-vibrations", split="train")

In [2]:
ds

Dataset({
    features: ['sample_id', 'segmented_overhead_file_name', 'speckle_vibrations_file_name', 'speckle_shifts_ifft_audio_file_name', 'audio_file_name', 'experiment_id', 'speakers', 'x_position', 'y_position', 'x_com', 'y_com', 'object', 'n_objects', 'box_material', 'mask_file_name', 'experiment_dir', 'manifest'],
    num_rows: 461
})

In [3]:
ex = ds[0]
ex['manifest']

'{"sample_id": 1, "experiment_id": "cube-00x01y_0001--31-03-18-21-24", "experiment_dir": "experiment-16", "source_experiment_id": "cube-00x01y_0001--31-03-18-21-24", "source_experiment_dir": "/net/mraid20/ifs/wisdom/groups/mark_sheinin_lab/DATA/experiment-15/cube-00x01y_0001--31-03-18-21-24", "hf_repo": "eturok-weizmann/laser-vibrations", "sample": {"object": "cube", "n_objects": 1, "box_material": "cardboard", "speakers": "0001", "x_position": 0, "y_position": 1, "image_dir": "cube-000x-001y-1obj-cardboard-2026-03-31-18-21-24"}, "segmentation": {"x_com": 277.91262776129247, "y_com": 136.59004286185296, "status": "completed"}, "experiment_config": {"audio": {"file_name": "audio/chirp_50_1000_3.0sec.wav", "sample_rate_hz": 44100, "duration_s": 3.2, "total_output_channels": 8, "wav_channels": 1, "sample_width_bytes": 2, "generation": {"signal": "chirp", "method": "logarithmic", "chirp_duration_s": 3.0, "silence_start_s": 0.1, "silence_end_s": 0.1, "f_start_hz": 50, "f_end_hz": 1000, "out

In [4]:
import json
artifacts = json.loads(ex['manifest'])['artifacts']
mask_path, fft_path = artifacts['mask_npz'], artifacts['speckle_shifts_fft']

In [5]:
def load_sample(ex):
    artifacts = json.loads(ex['manifest'])['artifacts']
    mask_path, fft_path = f"data/{artifacts['mask_npz']}", artifacts['speckle_shifts_fft']
    return dict(mask_path=mask_path, fft_path=fft_path)

In [6]:
ex

{'sample_id': 1,
 'segmented_overhead_file_name': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1451x1338>,
 'speckle_vibrations_file_name': <torchcodec.decoders._video_decoder.VideoDecoder at 0x1253f9b20>,
 'speckle_shifts_ifft_audio_file_name': <datasets.features._torchcodec.AudioDecoder at 0x11a194dd0>,
 'audio_file_name': <datasets.features._torchcodec.AudioDecoder at 0x1258676e0>,
 'experiment_id': 'cube-00x01y_0001--31-03-18-21-24',
 'speakers': '0001',
 'x_position': 0,
 'y_position': 1,
 'x_com': 277.91262776129247,
 'y_com': 136.59004286185296,
 'object': 'cube',
 'n_objects': 1,
 'box_material': 'cardboard',
 'mask_file_name': <PIL.PngImagePlugin.PngImageFile image mode=L size=1069x956>,
 'experiment_dir': 'experiment-16',
 'manifest': '{"sample_id": 1, "experiment_id": "cube-00x01y_0001--31-03-18-21-24", "experiment_dir": "experiment-16", "source_experiment_id": "cube-00x01y_0001--31-03-18-21-24", "source_experiment_dir": "/net/mraid20/ifs/wisdom/groups/mark_sheinin_l

In [7]:
import os

import torch
import numpy as np
from torch.utils.data import Dataset
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download

class VibrationDataset(Dataset):
    def __init__(self, repo_id:str, disc_mask_h:int, disc_mask_w:int, speakers:list[int,str]|list[int]|list[str]|str|None=None, n_objects:list[int]|int|None=None, n_samples:int=None, num_proc:int=8):
        self.ds = load_dataset("eturok-weizmann/laser-vibrations", split="train", num_proc=num_proc) # this is `data/metadata.jsonl`
        self.ds = self.ds.remove_columns(['segmented_overhead_file_name', 'speckle_vibrations_file_name', 'speckle_shifts_ifft_audio_file_name', 'audio_file_name', 'mask_file_name'])
        print(f"Loaded dataset with {len(self.ds)} samples")

        # Filter the dataset
        if speakers is not None:
            self.ds = self.ds.filter(lambda row: row["speakers"] in (str(speakers) if isinstance(speakers, list) else [speakers]), num_proc=num_proc)
            print(f"Filtered dataset to {len(self.ds)} samples with speakers={speakers}")
        if n_objects is not None:
            self.ds = self.ds.filter(lambda row: row["n_objects"] in (n_objects if isinstance(n_objects, list) else [n_objects]), num_proc=num_proc)
            print(f"Filtered dataset to {len(self.ds)} samples with n_objects={n_objects}")
        if n_samples is not None:
            self.ds = self.ds.select(range(min(n_samples, len(self.ds))))
            print(f"Selected first {n_samples} samples from the dataset")
        print(f"Final dataset contains {len(self.ds)} samples")

        # Download the segmentation masks and speckle shift FFTs for the filtered dataset
        print('Downloading masks and FFTs...')
        def get_path(paths): return [json.loads(manifest)['artifacts'][paths] for manifest in list(self.ds['manifest'])]
        mask_paths, fft_paths = get_path('mask_npz'), get_path('speckle_shifts_fft')
        print(f"Mask paths: {mask_paths[:5]}...\nFFT paths: {fft_paths[:5]}...")
        snapshot_dir = snapshot_download(repo_id, repo_type="dataset", allow_patterns=set(mask_paths+fft_paths)) # might be duplicate paths for masks
        print(f"Downloaded snapshot to {snapshot_dir}")

        # Load the masks and FFTs
        print('Loading masks and FFTs...')
        def load_sample(paths, key): return torch.stack([torch.from_numpy(np.load(os.path.join(snapshot_dir, path))[key]) for path in paths])
        self.masks, self.fft = load_sample(mask_paths, 'mask'), load_sample(fft_paths, 'fft')
        print(f"masks.shape={self.masks.shape}\nfft.shape={self.fft.shape}")

        # discretize masks and cast to float
        print('Discretizing masks...')
        self.masks = F.adaptive_avg_pool2d(self.masks[:, None].float(), (disc_mask_h, disc_mask_w)).squeeze()
        print(f"masks.shape={self.masks.shape}")

    def __len__(self): return len(self.ds)
    def __getitem__(self, idx): return dict(mask=self.masks[idx], fft=self.fft[idx])


disc_mask_h, disc_mask_w = 40, 20
ds = VibrationDataset("eturok-weizmann/laser-vibrations", disc_mask_h, disc_mask_w, speakers=[1000], n_objects=[0,1], n_samples=5)
ds

Loaded dataset with 461 samples
Filtered dataset to 115 samples with speakers=[1000]
Filtered dataset to 115 samples with n_objects=[0, 1]
Selected first 5 samples from the dataset
Final dataset contains 5 samples
Mask paths: ['image/cube-000x-001y-1obj-cardboard-2026-03-31-18-21-18/mask.npz', 'image/cube-000x-002y-1obj-cardboard-2026-03-31-18-27-07/mask.npz', 'image/cube-000x-003y-1obj-cardboard-2026-03-31-18-33-14/mask.npz', 'image/cube-000x-004y-1obj-cardboard-2026-03-31-18-39-23/mask.npz', 'image/cube-000x-005y-1obj-cardboard-2026-03-31-18-51-21/mask.npz']...
FFT paths: ['data/0000004/speckle_shifts_fft.npz', 'data/0000008/speckle_shifts_fft.npz', 'data/0000012/speckle_shifts_fft.npz', 'data/0000016/speckle_shifts_fft.npz', 'data/0000020/speckle_shifts_fft.npz']...


Fetching ... files: 0it [00:00, ?it/s]

Downloaded snapshot to /Users/eitanturok/.cache/huggingface/hub/datasets--eturok-weizmann--laser-vibrations/snapshots/331b7943a4106cffeceb36c8115c58a50a4b01f5
Loading masks and FFTs...
masks.shape=torch.Size([5, 956, 1069])
fft.shape=torch.Size([5, 100, 3421, 2])
Discretizing masks...
masks.shape=torch.Size([5, 40, 20])


In [8]:
ds.masks.shape, ds.fft.shape

(torch.Size([5, 40, 20]), torch.Size([5, 100, 3421, 2]))